# Using `matcher.py`

This notebook shows how to import and use `AdaptiveTemplateMatching` from the packaged module in `src/adaptive_template_matching/matcher.py`.

Before running the examples, install the package from the repository root:

```bash
pip install -e git+https://github.com/Garangatang/adaptive-template-matching-repo/tree/main
```

## 1. Import the matcher

The package exposes `AdaptiveTemplateMatching` at the top level.

In [ ]:
from adaptive_template_matching import AdaptiveTemplateMatching
import numpy as np

## 2. Import example data from .pkl file

Upsamp_UP_Dict.pkl contains underfoot pressure data from six participants utilized in the paper associated with this repo.

In [ ]:
def openPkl(file_path):
    with open(f"{file_path}.pkl", 'rb') as f:
        return_data = pkl.load(f)

    return return_data

def saveToPkl(file_path, data):
    with open(f"{file_path}.pkl", "wb") as f:
        pkl.dump(data, f)

In [ ]:
dataDict = openPkl("data/Upsamp_UP_Dict.pkl")

## 3. Create a matcher instance

You can use the defaults designed for GRF data sampled at 1980 Hz or customize the template geometry.

If you are using a dataset with a different sample rate, use the equation:
TEMPLATE_LEN = (200 / 1980) * YOUR_DATA_SAMPLE_RATE

In [ ]:
# Template construction parameters, based on parameters used for the associated paper
CHANGE_POINT_ARR = [125, 180]
ANGLE_ARR = [80, 85]
TEMPLATE_LEN = 200
BASELINE = 0
TEMPLATE_SCALER = 0.12
REFLECT = False

global_matcher = AdaptiveTemplateMatching(
    cp_inds=CHANGE_POINT_ARR,
    template_angls=ANGLE_ARR,
    template_len=TEMPLATE_LEN,
    baseline=BASELINE,
    template_scaler=TEMPLATE_SCALER,
    reflect=REFLECT,
)

## 4. Run a scan

Tune the thresholds to match your signal scaling and data quality.

Available options for run functions are: cold_start_run_dataset, warm_start_run_dataset, and all_data_run_dataset.

Data input into the adaptive template matching must be input as a dictionary in format of: 
key: dataset name, value: 1D numpy array of data.



In [ ]:
# Analyze the given example data for structure if curious
DATA_DICT = openPkl("data/Upsamp_UP_Dict")

In [ ]:
# Thresholds/gates
AMP_MAX = 0.30
THR_PASS = 0.80
REL_ERR = 0.40
PASSES = 2

# Sum absolute difference controls
POS_SHIFT = 30
TEMPLATE_LEN = 200
NSAD_THRESH     = int(-0.3 * TEMPLATE_LEN)
MIN_STD        = 1e-3

# Low-amplitude gating
ENFORCE_LOW_AMP = True
LOW_AMP_QUANTILE = 0.35
LOW_AMP_COVER     = 0.50

# Clustering
CLUSTER_BY = "sad"   # available options: {"r", "sad", "sad_shifted"}
CLUSTER_RADIUS = 1000    # at 1980 Hz, set per expected inter-event spacing

# Diagnostics
SHOW_DEBUG     = False
DEBUG_VERBOSE  = False


In [ ]:
"""
A cold start run restarts the template to the piecewise linear function for each input 
dataset before updating the template and parsing.
"""

cold_start_dict = global_matcher.cold_start_run_dataset(data_dict=DATA_DICT,
        amp_max = AMP_MAX, thr = THR_PASS, rel_err = REL_ERR, passes = PASSES,
        pos_shift = POS_SHIFT, sad_thresh = NSAD_THRESH, min_std = MIN_STD,
        enforce_low_amp=ENFORCE_LOW_AMP,
        low_amp_quantile=LOW_AMP_QUANTILE,
        low_amp_cover=LOW_AMP_COVER,
        cluster_metric=CLUSTER_BY,
        cluster_radius=CLUSTER_RADIUS,
        show_debug=SHOW_DEBUG, debug_verbose=DEBUG_VERBOSE)

In [ ]:
"""
A warm start run updates the template from one dataset, then utilizes that updated template to parse all remaining datasets without 
updating the template further.

TRAIN_DATA_SET set to None means a random dataset will be chosen from the given DATA_DICT to perform the update. If you wish to update 
from a specific dataset, set TRAIN_DATA_SET to the dictionary key matching the desired dataset.
"""
TRAIN_DATA_SET = None
warm_start_dict = global_matcher.warm_start_run_dataset(data_dict=DATA_DICT, name=TRAIN_DATA_SET
        amp_max = AMP_MAX, thr = THR_PASS, rel_err = REL_ERR, passes = PASSES,
        pos_shift = POS_SHIFT, sad_thresh = NSAD_THRESH, min_std = MIN_STD,
        enforce_low_amp=ENFORCE_LOW_AMP,
        low_amp_quantile=LOW_AMP_QUANTILE,
        low_amp_cover=LOW_AMP_COVER,
        cluster_metric=CLUSTER_BY,
        cluster_radius=CLUSTER_RADIUS,
        show_debug=SHOW_DEBUG, debug_verbose=DEBUG_VERBOSE)

In [ ]:
"""
An all data run updates the template across all datasets n = PASSES times per dataset, where the template updates are passed from one 
dataset to the next before being updated further. This final template updated across all data is then used to parse all datasets.
"""
all_data_run_dict = global_matcher.all_data_run_dataset(data_dict=DATA_DICT,
        amp_max = AMP_MAX, thr = THR_PASS, rel_err = REL_ERR, passes = PASSES,
        pos_shift = POS_SHIFT, sad_thresh = NSAD_THRESH, min_std = MIN_STD,
        enforce_low_amp=ENFORCE_LOW_AMP,
        low_amp_quantile=LOW_AMP_QUANTILE,
        low_amp_cover=LOW_AMP_COVER,
        cluster_metric=CLUSTER_BY,
        cluster_radius=CLUSTER_RADIUS,
        show_debug=SHOW_DEBUG, debug_verbose=DEBUG_VERBOSE)

## 5. Helpers

In [ ]:
"""
Pull the template shape as a numpy array after being updated through one of three run modes above.
"""
template_shape = global_matcher.get_template_shape()

In [ ]:
"""
Pull the correlation scores for each found maximum overlap point between the given signal and template.
"""
corr_scores = global_matcher.get_final_idx_scores()

In [ ]:
"""
Pull the nsad signal to analyze time-series of maximum overlap points between the given signal and template.
"""
corr_scores = global_matcher.get_final_nsad_signal()

## Notes

- `scan_matches` is the lowest-level matching call.
- `final_template_match_and_plot` is useful when you want summary output and optional plots.
- `cold_start_run_dataset`, `warm_start_run_dataset`, and `all_data_run_dataset` are convenient for multi-dataset adaptation workflows.